In [1]:
import pandas as pd

commit = pd.read_csv("Commit.csv")

races = pd.read_csv("Nemo.csv")

forcedecks = pd.read_csv("Forcedeck.csv")

In [2]:
commit["athlete_id"] = commit["About"]
commit["timestamp"] = pd.to_datetime(commit["workout_date_time"], errors = "coerce")

races["athlete_id"] = races["AthleteName"]
races["race_date"] = pd.to_datetime(races["Date"], errors = "coerce")

C:\Users\adhamz\AppData\Local\Temp\ipykernel_42424\547141096.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  races["race_date"] = pd.to_datetime(races["Date"], errors = "coerce")


In [3]:
fd_cmj = forcedecks[forcedecks["Test Type"] == "CMJ"].copy()

fd_cmj["athlete_id"] = fd_cmj["About"]
fd_cmj["timestamp"] = pd.to_datetime(fd_cmj["Date"], dayfirst=True, utc=True, errors="coerce")

In [4]:
min_commit = commit["timestamp"].min()
races_in_range = races[pd.to_datetime(races["race_date"], utc = True) >= min_commit].copy()

In [5]:
import numpy as np

commit["session_day"] = commit["timestamp"].dt.floor("D")
commit["session_block"] = np.where(commit["timestamp"].dt.hour < 12, "AM", "PM")

commit = commit.drop_duplicates(subset=["athlete_id", "session_day", "session_block", "workout_total_distance"])

In [6]:
Intensity_Cols = ["intensity_anaerobic_capacity", "intensity_aerobic_power", "intensity_anaerobic_power", "intensity_speed"]

Stroke_map = {
    "Freestyle": "stroke_free",
    "Backstroke": "stroke_back",
    "Breaststroke": "stroke_breast",
    "Butterfly": "stroke_fly",
    "Medley": "stroke_im",}

In [7]:
races_in_range["race_stroke"] = races_in_range["RaceEventSwimmingStyleName"]

In [8]:
def training_distance_features(commit, athlete_id, race_date, race_stroke):

    race_date = pd.to_datetime(race_date, utc = True)
    df = commit[commit["athlete_id"] == athlete_id]

    features = {}

    stroke_col = Stroke_map.get(race_stroke)

    for days in [10, 40]:
        start = race_date - pd.Timedelta(days=days)
        w = df[(df["timestamp"] >= start ) & (df["timestamp"] < race_date)]

        features[f"distance_{days}d"] = w["workout_total_distance"].sum()
        features[f"sessions_{days}d"] = w["workout_total_distance"].count()

        for col in Intensity_Cols:
            if col in w.columns:
                features[f"{col}_{days}d"] = w[col].sum()
            else:
                features[f"{col}_{days}d"] = 0

        #Stroke specific metres (dependant on race stroke)

        if stroke_col and stroke_col in w.columns:
            features[f"distance_{race_stroke}_{days}d"] = w[stroke_col].sum()
        else:
            features[f"distance_{race_stroke}_{days}d"] = 0

        #Taper ratio:
        d10 = features.get("distance_10d", 0)
        d40 = features.get("distance_40d", 0)

        features["taper_ratio"] = round((d10 / d40) * 100, 2) if d40 not in [0, None] else 0

    return features

In [9]:
def race_row(r):
    features = training_distance_features(commit, r["athlete_id"], r["race_date"], r["race_stroke"])
    
    features["race_phase"] = r["PhaseName"]
    features["race_stroke"] = r["RaceEventSwimmingStyleName"]
    features["race_distance"] = r["RaceEventDistance"]
    features["pool_length"] = r["PoolLength"]
    features["race_time"] = r["RaceTime"]
    
    cmj_df = fd_cmj[fd_cmj["athlete_id"] == r["athlete_id"]]
    r["race_date"] = pd.to_datetime(r["race_date"], utc=True)
    
    for days in [10, 40]:
        start = r["race_date"] - pd.Timedelta(days=days)
    
        w_cmj = cmj_df[
            (cmj_df["timestamp"] >= start) &
            (cmj_df["timestamp"] < r["race_date"])
        ]
        
        features[f"cmj_tests_{days}d"] = len(w_cmj)
        features[f"cmj_jump_height_mean_{days}d"] = w_cmj["Jump Height (Flight Time) [cm]"].mean()
        features[f"cmj_jump_height_max_{days}d"] = w_cmj["Jump Height (Flight Time) [cm]"].max()
    
        features[f"cmj_peak_power_mean_{days}d"] = w_cmj["Peak Power / BM [W/kg]"].mean()
        features[f"cmj_peak_power_max_{days}d"] = w_cmj["Peak Power / BM [W/kg]"].max()
    
        features[f"cmj_concentric_force_mean_{days}d"] = w_cmj["Concentric Mean Force [N]"].mean()
        features[f"cmj_concentric_force_max_{days}d"] = w_cmj["Concentric Mean Force [N]"].max()
    
        features[f"cmj_eccentric_force_mean_{days}d"] = w_cmj["Eccentric Mean Force [N]"].mean()
        features[f"cmj_eccentric_force_max_{days}d"] = w_cmj["Eccentric Mean Force [N]"].max()

        return features


In [17]:

r = races_in_range.iloc[0].copy()
features = race_row(r)
f2 = dict()
for k in features.keys():
    f2[k] = [features[k]]
training1 = pd.DataFrame(f2)


for i in range(1, len(races_in_range)):
    r = races_in_range.iloc[i].copy()
    features = race_row(r)
    f2 = dict()
    for k in features.keys():
        f2[k] = [features[k]]
    nextrow = pd.DataFrame(f2)
    training1 = pd.concat([training1, nextrow], ignore_index=True)
training1

,distance_10d,sessions_10d,intensity_anaerobic_capacity_10d,intensity_aerobic_power_10d,intensity_anaerobic_power_10d,intensity_speed_10d,distance_Freestyle_10d,taper_ratio,distance_40d,sessions_40d,...,cmj_eccentric_force_mean_10d,cmj_eccentric_force_max_10d,distance_Backstroke_10d,distance_Backstroke_40d,distance_Medley_10d,distance_Medley_40d,distance_Butterfly_10d,distance_Butterfly_40d,distance_Breaststroke_10d,distance_Breaststroke_40d
0,50225,11,1100.0,100.0,100.0,350.0,40570.0,29.52,170125,41,...,963.160333,964.297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,0,0.0,0.0,0.0,0.0,NaN,0.00,0,0,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,54450,12,950.0,600.0,200.0,70.0,45300.0,25.84,210740,48,...,800.539667,801.810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,43345,12,50.0,1500.0,1050.0,405.0,NaN,19.93,217470,51,...,825.316333,825.583,NaN,NaN,700.0,1900.0,NaN,NaN,NaN,NaN
4,31450,12,750.0,300.0,700.0,375.0,27300.0,18.24,172430,50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
748,56500,14,225.0,500.0,500.0,765.0,44000.0,24.16,233810,54,...,786.913000,788.757,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
749,42510,13,600.0,0.0,280.0,350.0,NaN,21.71,195820,51,...,683.778500,688.372,NaN,NaN,NaN,NaN,1250.0,5275.0,NaN,NaN
750,45650,11,1600.0,0.0,0.0,675.0,NaN,32.67,139725,42,...,639.016667,639.969,NaN,NaN,NaN,NaN,75.0,1600.0,NaN,NaN
751,0,0,0.0,0.0,0.0,0.0,NaN,0.00,0,0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN


In [13]:
races_in_range.iloc[750]

RaceId                    fe368004-6f8d-4f57-8696-00c3cf9af56d
PoolLength                                                  50
PhaseNumber                                                1.0
Date                                                19/11/2025
Lane                                                       4.0
                                          ...                 
Turn58BreakoutDistance                                     NaN
Turn59BreakoutDistance                                     NaN
athlete_id                                     Keanna MacInnes
race_date                                  2025-11-19 00:00:00
race_stroke                                          Butterfly
Name: 2822, Length: 765, dtype: object

In [19]:
target = "race_time"

In [20]:
x = training1.drop(columns=[target])
y = training1[target]